# Interpreting GPT4TS / One Fits All with WinTSR

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/khairulislam/tslens/blob/main/notebooks/pretrained_gpt4ts.ipynb)

[One Fits All (GPT4TS)](https://arxiv.org/abs/2302.11939) repurposes a frozen pretrained GPT-2 backbone for time-series tasks. Unlike Timer, MOMENT, or TTM, GPT-2 is not an end-to-end forecaster: we must quickly train GPT4TS's patch embedding and forecast head before interpreting it.

In [ ]:
%pip install -q tslens einops transformers
!git clone -q --depth 1 https://github.com/thuml/OpenLTM.git
%cd OpenLTM

## 1. Build the frozen-backbone forecaster

OpenLTM loads pretrained `gpt2`, keeps only the first `gpt_layers`, and freezes every GPT-2 parameter except names containing `ln` or `wpe`. The time-series `in_layer` and `out_layer` remain trainable. That split is the paper's central idea: preserve the general-purpose Transformer blocks and learn a thin task adapter.

In [ ]:
from types import SimpleNamespace
import torch
from models import gpt4ts

torch.manual_seed(0)
SEQ_LEN, PRED_LEN = 96, 96
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
configs = SimpleNamespace(
    seq_len=SEQ_LEN, patch_size=16, stride=8, gpt_layers=3,
    test_pred_len=PRED_LEN, d_model=768, dropout=0.1, use_norm=True,
)
model = gpt4ts.Model(configs).to(device)

assert all(p.requires_grad == ("ln" in n or "wpe" in n) for n, p in model.gpt2.named_parameters())
counts = {
    "frozen GPT-2": sum(p.numel() for n, p in model.named_parameters() if n.startswith("gpt2.") and not p.requires_grad),
    "trainable GPT-2 ln/wpe": sum(p.numel() for n, p in model.named_parameters() if n.startswith("gpt2.") and p.requires_grad),
    "trainable in/out layers": sum(p.numel() for n, p in model.named_parameters() if n.startswith(("in_layer.", "out_layer."))),
}
for name, count in counts.items():
    print(f"{name:<28} {count / 1e6:6.2f}M")

## 2. Fit the forecasting layers on ETTh2

We turn the real ETTh2 oil-temperature series into 96-to-96 sliding windows. This is a small supervised adaptation, not zero-shot forecasting. The loop mirrors OpenLTM's essentials: trainable parameters only, Adam, MSE, and evaluation without gradients.

In [ ]:
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset

DATA_URL = "https://raw.githubusercontent.com/WenWeiTHU/TimeSeriesDatasets/refs/heads/main/ETT-small/ETTh2.csv"
series = torch.tensor(pd.read_csv(DATA_URL)["OT"].dropna().to_numpy(), dtype=torch.float32)

def make_windows(values, step=8):
    windows = values.unfold(0, SEQ_LEN + PRED_LEN, step)
    return windows[:, :SEQ_LEN].unsqueeze(-1), windows[:, SEQ_LEN:].unsqueeze(-1)

x_train, y_train = make_windows(series[:4096])
x_valid, y_valid = make_windows(series[4096 - SEQ_LEN : 5200])
train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=32, shuffle=True)
valid_loader = DataLoader(TensorDataset(x_valid, y_valid), batch_size=64)
print("train:", tuple(x_train.shape), tuple(y_train.shape), "valid:", tuple(x_valid.shape), tuple(y_valid.shape))

In [ ]:
def forward_batch(x):
    x_mark = x.new_zeros(len(x), SEQ_LEN, 4)
    y_mark = x.new_zeros(len(x), PRED_LEN, 4)
    return model(x, x_mark, y_mark)

def evaluate(loader):
    model.eval()
    total, count = 0.0, 0
    with torch.inference_mode():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            total += torch.nn.functional.mse_loss(forward_batch(x), y, reduction="sum").item()
            count += y.numel()
    return total / count

trainable = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.Adam(trainable, lr=1e-3)
loss_fn = torch.nn.MSELoss()
initial_mse = evaluate(valid_loader)
print(f"initial validation MSE: {initial_mse:.4f}")

for epoch in range(4):
    model.train()
    running = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = loss_fn(forward_batch(x), y)
        loss.backward()
        optimizer.step()
        running += loss.item() * len(x)
    print(f"epoch {epoch + 1}: train MSE {running / len(x_train):.4f} | valid MSE {evaluate(valid_loader):.4f}")

final_mse = evaluate(valid_loader)
print(f"validation improvement: {(initial_mse - final_mse) / initial_mse:.1%}")

## 3. Check a held-out forecast

The loss and forecast should improve during the short fit. This is a tutorial-scale adaptation, not a benchmark result.

In [ ]:
import matplotlib.pyplot as plt

x_test = x_valid[:1].to(device)
ground_truth = y_valid[0, :, 0]
model.eval()
with torch.inference_mode():
    prediction = forward_batch(x_test)[0, :, 0].detach().cpu()

plt.figure(figsize=(12, 4))
plt.plot(torch.arange(-SEQ_LEN, 0), x_test[0, :, 0].cpu(), label="lookback", linewidth=1)
plt.plot(torch.arange(PRED_LEN), ground_truth, label="ground truth", linewidth=2)
plt.plot(torch.arange(PRED_LEN), prediction, label="GPT4TS forecast", linewidth=2)
plt.axvline(0, color="0.4", linestyle="--", linewidth=1)
plt.xlabel("hour relative to forecast")
plt.ylabel("oil temperature (OT)")
plt.title("Adapted GPT4TS forecast on held-out ETTh2 data")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

## 4. Explain the adapted model

GPT4TS already maps one `(batch, seq_len, n_features)` tensor to one forecast tensor, so it uses tslens's single-input convention. Its `x_mark` and `y_mark` arguments are present in `forward()` but unused by the OpenLTM implementation; zero calendar tensors satisfy the signature and remain fixed as `additional_forward_args`.

In [ ]:
from tslens import WinTSR

x_enc = x_valid[:2].to(device)
x_mark = torch.zeros(len(x_enc), SEQ_LEN, 4, device=device)
y_mark = torch.zeros(len(x_enc), PRED_LEN, 4, device=device)
attr = WinTSR(model.eval()).attribute(
    inputs=x_enc,
    baselines=torch.zeros_like(x_enc),
    additional_forward_args=(x_mark, y_mark),
    threshold=0.5,
    show_progress=True,
)
print("attributions:", tuple(attr.shape), "= (batch, horizon, time, feature)")

## 5. See which time steps mattered

The input and attribution share an x-axis, matching the other pretrained-model tutorials. We average absolute attribution over all 96 forecast horizons for sample 0.

In [ ]:
recent = x_enc[0, :, 0].detach().cpu()
saliency = attr[0].abs().mean(dim=0).squeeze(-1).detach().cpu()
steps = torch.arange(-SEQ_LEN, 0)

fig, axes = plt.subplots(2, 1, figsize=(12, 4), sharex=True, height_ratios=(2, 1))
axes[0].plot(steps, recent, color="tab:blue", linewidth=1)
axes[0].set_ylabel("OT")
axes[0].set_title("GPT4TS input and WinTSR attribution")
axes[0].grid(alpha=0.25)
axes[1].fill_between(steps, saliency, color="tab:orange", alpha=0.8)
axes[1].plot(steps, saliency, color="tab:orange", linewidth=0.8)
axes[1].set_xlabel("hours before forecast")
axes[1].set_ylabel("attribution")
axes[1].grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 6. Frozen does not mean opaque

One Fits All argues that a largely frozen general-purpose backbone can transfer across time-series tasks through small learned interfaces. WinTSR asks a different question: which input replacements change this fitted forecaster's outputs? Its default occlusion backend needs no gradient access through GPT-2, so attribution works the same way whether the backbone weights are frozen or trainable.

## Next steps

- Train longer or use more ETTh2 windows, then verify improvements on a strictly later test split.
- Try a seasonal or local-mean attribution baseline instead of zero.
- Add several ETTh2 channels to compare time and feature relevance.

If this was useful, please star the [repository](https://github.com/khairulislam/tslens). Please cite the following if you use our work:

```bibtex
@inproceedings{zhou2023onefitsall,
  title={One Fits All: Power General Time Series Analysis by Pretrained LM},
  author={Zhou, Tian and Niu, Peisong and Wang, Xue and Sun, Liang and Jin, Rong},
  booktitle={Advances in Neural Information Processing Systems},
  year={2023},
  eprint={2302.11939}
}

@article{islam2024wintsr,
  title={WinTSR: A Windowed Temporal Saliency Rescaling Method for Interpreting Time Series Deep Learning Models},
  author={Islam, Md Khairul and Fox, Judy},
  journal={arXiv preprint arXiv:2412.04532},
  year={2024}
}
```